# Flood LSTM – 4 Main Reservoirs (Quảng Nam)

Train **per-reservoir** models cho 4 hồ điều tiết lũ chính:
- **A Vương** (upstream, fast-response)
- **DakMi4** (largest, main flood control)
- **Sông Bung 4** (Bung tributary)
- **Sông Tranh 2** (lower basin)

**Architecture**: Bi-LSTM Encoder + Multi-Head Attention + Autoregressive Decoder  
**Output**: 24h forecast × 3 quantiles [P10, P50, P90]  
**Input**: 240h (10 days) historical features × 46 features

### Dataset structure (Kaggle input):
```
/kaggle/input/flood-lstm-4reservoirs/
  avuong/      X_past.npy  X_future.npy  y.npy  timestamps.npy
  dakmi4/      ...
  songbung4/   ...
  songtranh2/  ...
```

In [ ]:
import os, math, time, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

print('PyTorch:', torch.__version__)
DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
print('Device :', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU    :', torch.cuda.get_device_name(0))

## 1. Configuration

In [ ]:
# ── Auto-detect DATA_DIR ──────────────────────────────────────────────────────
import os

print("=== /kaggle/input/ contents ===")
if os.path.exists('/kaggle/input'):
    for _d in sorted(os.listdir('/kaggle/input')):
        _p = os.path.join('/kaggle/input', _d)
        if os.path.isdir(_p):
            _sub = sorted(os.listdir(_p))
            print(f"  {_d}/  ->  {_sub[:8]}")
else:
    print("  /kaggle/input KHONG TON TAI")

DATA_DIR = None
for _root, _dirs, _files in os.walk('/kaggle/input'):
    if 'avuong' in _dirs:
        DATA_DIR = _root
        break

if DATA_DIR is None:
    raise RuntimeError(
        "Khong tim thay 'avuong/' trong /kaggle/input.\n"
        "Kiem tra: dataset da Add vao notebook chua?"
    )

print(f"\nDATA_DIR : {DATA_DIR}")
OUT_DIR = '/kaggle/working'

print("\nDataset structure:")
for _item in sorted(os.listdir(DATA_DIR)):
    _p = os.path.join(DATA_DIR, _item)
    if os.path.isdir(_p):
        _npy = [f for f in os.listdir(_p) if f.endswith('.npy')]
        print(f"  {_item}/  ->  {_npy}")
    else:
        print(f"  {_item}")

# ── Per-reservoir configs (v4) ────────────────────────────────────────────────
RESERVOIR_CONFIGS = {
    'avuong': {
        'display_name'   : 'HO A VUONG',
        'hidden_size'    : 192,
        'horizon'        : 24,
        'batch_size'     : 512,
        'epochs'         : 120,
        'lr'             : 2e-4,
        'weight_decay'   : 2e-3,
        'warmup_epochs'  : 8,
        'patience'       : 40,
        'quantiles'      : [0.1, 0.5, 0.9],
        'n_future'       : 3,
        'inflow_cap_sqrt': 50.0,
    },
    'dakmi4': {
        'display_name'   : 'HO DAK MI 4',
        'hidden_size'    : 192,
        'horizon'        : 24,
        'batch_size'     : 512,
        'epochs'         : 150,
        'lr'             : 2e-4,   # tang tu 1e-4 de thoat local minimum som
        'weight_decay'   : 3e-3,
        'warmup_epochs'  : 10,
        'patience'       : 50,    # tang tu 35 cho DakMi4 co nhieu thoi gian hon
        'quantiles'      : [0.1, 0.5, 0.9],
        'n_future'       : 3,
        'inflow_cap_sqrt': 67.08,
    },
    'songbung4': {
        'display_name'   : 'HO SONG BUNG 4',
        'hidden_size'    : 192,
        'horizon'        : 24,
        'batch_size'     : 512,
        'epochs'         : 120,
        'lr'             : 2e-4,
        'weight_decay'   : 2e-3,
        'warmup_epochs'  : 8,
        'patience'       : 40,
        'quantiles'      : [0.1, 0.5, 0.9],
        'n_future'       : 3,
        'inflow_cap_sqrt': 67.08,
    },
    'songtranh2': {
        'display_name'   : 'HO SONG TRANH 2',
        'hidden_size'    : 192,
        'horizon'        : 24,
        'batch_size'     : 512,
        'epochs'         : 120,
        'lr'             : 2e-4,
        'weight_decay'   : 2e-3,
        'warmup_epochs'  : 8,
        'patience'       : 40,
        'quantiles'      : [0.1, 0.5, 0.9],
        'n_future'       : 3,
        'inflow_cap_sqrt': 89.44,
    },
}

print('\nReservoirs to train:', list(RESERVOIR_CONFIGS.keys()))

## 2. Model Architecture

In [ ]:
class PerReservoirModel(nn.Module):
    """
    Bi-LSTM Encoder + Multi-Head Attention + Autoregressive Decoder.
    Không dùng reservoir embedding (model riêng cho từng hồ).
    
    x_past  : (B, 240, 46)  ->  lịch sử 10 ngày, đã StandardScale
    x_future: (B, 24, 3)    ->  dự báo mưa 24h tới (oracle khi train)
    output  : (B, 24, 3)    ->  [P10, P50, P90] trong sqrt space
    """

    def __init__(self, input_size, hidden_size, horizon, quantiles, n_future=3):
        super().__init__()
        self.horizon  = horizon
        self.num_q    = len(quantiles)
        self.n_future = n_future

        self.hindcast_proj = nn.Linear(input_size, 32)

        self.encoder = nn.LSTM(
            32, hidden_size, num_layers=2,
            dropout=0.2, batch_first=True, bidirectional=True,
        )
        self.enc_proj = nn.Linear(hidden_size * 2, hidden_size)

        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_size, num_heads=4, dropout=0.1, batch_first=True,
        )

        self.future_emb = nn.Linear(n_future, 16) if n_future > 0 else None
        future_dim      = 16 if n_future > 0 else 0

        self.handoff = nn.Sequential(
            nn.Linear(hidden_size * 4, hidden_size * 2),
            nn.LayerNorm(hidden_size * 2),
            nn.ReLU(),
            nn.Dropout(0.2),
        )
        self.handoff_out = nn.Linear(hidden_size * 2, hidden_size * 2)

        self.decoder = nn.LSTM(
            hidden_size + self.num_q + future_dim,
            hidden_size, num_layers=2, dropout=0.2, batch_first=True,
        )
        self.dropout = nn.Dropout(0.2)
        self.fc      = nn.Linear(hidden_size, self.num_q)

        nn.init.xavier_uniform_(self.hindcast_proj.weight)
        nn.init.xavier_uniform_(self.handoff_out.weight)
        if self.future_emb is not None:
            nn.init.xavier_uniform_(self.future_emb.weight)

    def forward(self, x_past, x_future=None):
        enc_in  = F.relu(self.hindcast_proj(x_past))
        enc_out, (h_enc, c_enc) = self.encoder(enc_in)
        enc_proj = self.enc_proj(enc_out)

        h_last = torch.cat([h_enc[-2], h_enc[-1]], dim=-1)
        c_last = torch.cat([c_enc[-2], c_enc[-1]], dim=-1)
        hc     = self.handoff_out(self.handoff(torch.cat([h_last, c_last], dim=-1)))
        h_dec, c_dec = hc.chunk(2, dim=-1)
        h_dec = h_dec.unsqueeze(0).repeat(2, 1, 1).contiguous()
        c_dec = c_dec.unsqueeze(0).repeat(2, 1, 1).contiguous()

        B     = x_past.size(0)
        prev_q = torch.zeros(B, 1, self.num_q, device=x_past.device)
        outputs = []

        for t in range(self.horizon):
            query   = h_dec[-1].unsqueeze(1)
            ctx, _  = self.attention(query, enc_proj, enc_proj)

            if self.future_emb is not None and x_future is not None:
                fut_emb = F.relu(self.future_emb(x_future[:, t:t+1, :]))
                dec_in  = torch.cat([ctx, prev_q, fut_emb], dim=2)
            else:
                dec_in = torch.cat([ctx, prev_q], dim=2)

            dec_out, (h_dec, c_dec) = self.decoder(dec_in, (h_dec, c_dec))
            step_q = F.softplus(self.fc(self.dropout(dec_out)))
            outputs.append(step_q)
            prev_q = step_q

        out = torch.cat(outputs, dim=1)
        return torch.sort(out, dim=2).values

print('PerReservoirModel defined.')

## 3. Loss Function

In [ ]:
def quantile_loss(preds, target, qs):
    """
    Hybrid: 55% Pinball + 20% Peak-MSE + 25% NSE-Loss (guarded).
    NSE-Loss chỉ compute khi batch có đủ variance (tránh gradient noise).
    """
    if not isinstance(qs, torch.Tensor):
        qs = torch.tensor(qs, device=preds.device, dtype=preds.dtype)

    # ── Pinball loss ──────────────────────────────────────────────────────────
    errors = target.unsqueeze(-1) - preds
    q_loss = torch.max((qs - 1) * errors, qs * errors)
    bias = torch.ones_like(q_loss)
    bias[:, :, 1] *= 1.2                                  # P50 bias
    q_loss = q_loss * bias
    hw = torch.exp(-0.02 * torch.arange(
        preds.shape[1], device=preds.device, dtype=preds.dtype))
    q_loss = (q_loss * hw.unsqueeze(0).unsqueeze(-1)).mean()

    # ── Peak-weighted MSE (sqrt space) ───────────────────────────────────────
    med = preds[:, :, 1]
    w   = torch.sqrt(target + 1.0)
    w   = w / (w.mean() + 1e-8)
    w   = w.clamp(max=5.0)
    mse = (w * (med - target) ** 2).mean()

    # ── NSE Loss (original space) — guarded theo variance batch ──────────────
    # Chỉ áp dụng khi batch có ss_tot đủ lớn (std > ~5 m³/s cho 12k points)
    # Tránh noisy gradient khi batch toàn low-flow uniform
    p_raw    = med ** 2
    t_raw    = target ** 2
    ss_res   = torch.sum((t_raw - p_raw) ** 2)
    ss_tot   = torch.sum((t_raw - t_raw.mean()) ** 2)
    nse_raw  = (ss_res / (ss_tot + 1e-6)).clamp(max=3.0)
    # Guard: zero-out khi batch quá flat (ss_tot < 5e5 ≈ std < 6 m³/s@12k pts)
    nse_weight = (ss_tot.detach() > 5e5).float()
    nse_loss   = nse_raw * nse_weight

    return 0.55 * q_loss + 0.20 * mse + 0.25 * nse_loss


def compute_metrics(preds, targets):
    """MAE, RMSE, NSE trong original space (m³/s)."""
    p_raw  = preds[:, :, 1] ** 2
    t_raw  = targets ** 2
    mae    = torch.mean(torch.abs(p_raw - t_raw)).item()
    rmse   = torch.sqrt(torch.mean((p_raw - t_raw) ** 2)).item()
    ss_res = torch.sum((t_raw - p_raw) ** 2)
    ss_tot = torch.sum((t_raw - t_raw.mean()) ** 2)
    nse    = (1.0 - ss_res / (ss_tot + 1e-8)).item()
    return mae, rmse, nse

print('Loss functions defined  (Pinball 55% + Peak-MSE 20% + Guarded-NSE 25%)')

## 4. Dataset & DataLoader

In [ ]:
class ReservoirDataset(Dataset):
    def __init__(self, X_past, X_future, y, cap=None):
        self.X_past   = X_past
        self.X_future = X_future
        self.y        = y
        self.cap      = cap

    def __len__(self):
        return len(self.X_past)

    def __getitem__(self, idx):
        y = np.array(self.y[idx], copy=True).astype(np.float32)
        if self.cap is not None:
            y = np.clip(y, 0.0, self.cap)
        x_past = torch.from_numpy(np.array(self.X_past[idx], copy=False)).float()
        x_fut  = (
            torch.from_numpy(np.array(self.X_future[idx], copy=False)).float()
            if self.X_future is not None else torch.zeros(0)
        )
        return x_past, x_fut, torch.from_numpy(y).float()


def load_reservoir(name, data_dir, cfg):
    d = os.path.join(data_dir, name)
    X_past  = np.load(os.path.join(d, 'X_past.npy'),     mmap_mode='r')
    y       = np.load(os.path.join(d, 'y.npy'),          mmap_mode='r')
    ts      = np.load(os.path.join(d, 'timestamps.npy'), mmap_mode='r')
    xf_path = os.path.join(d, 'X_future.npy')
    X_fut   = np.load(xf_path, mmap_mode='r') if os.path.exists(xf_path) else None
    cap     = cfg['inflow_cap_sqrt']

    # ── Split ────────────────────────────────────────────────────────────────
    # Val  : flood season 2024 (Sep 2024 – Mar 2025)
    # Test : flood season 2025 (Oct – Dec 2025)  ← mục tiêu đánh giá chính
    # Train: tất cả còn lại (2022 → Aug 2024  +  Mar 2025 → Sep 2025)
    VAL_START  = np.datetime64('2024-09-01', 's')
    VAL_END    = np.datetime64('2025-03-01', 's')
    TEST_START = np.datetime64('2025-10-01', 's')
    TEST_END   = np.datetime64('2026-01-01', 's')

    all_idx = np.arange(len(ts))
    tr_idx  = all_idx[
        (ts < VAL_START) | ((ts >= VAL_END) & (ts < TEST_START))
    ]
    va_idx  = all_idx[(ts >= VAL_START) & (ts < VAL_END)]
    te_idx  = all_idx[(ts >= TEST_START) & (ts < TEST_END)]

    def ds(idx):
        xf = X_fut[idx] if X_fut is not None else None
        return ReservoirDataset(X_past[idx], xf, y[idx], cap)

    train_ds = ds(tr_idx)
    val_ds   = ds(va_idx)
    test_ds  = ds(te_idx)

    # Flood oversampling: top 5% ×2, top 1% ×3
    peaks  = np.array(y[tr_idx]).max(axis=1)
    t95    = float(np.percentile(peaks, 95))
    t99    = float(np.percentile(peaks, 99))
    i95    = np.where(peaks >= t95)[0].tolist()
    i99    = np.where(peaks >= t99)[0].tolist()
    os_idx = list(range(len(train_ds))) + i95 * 2 + i99 * 3
    train_os = Subset(train_ds, os_idx)

    has_future = X_fut is not None
    print(f'  Train {len(tr_idx):,} -> OS {len(os_idx):,} | Val {len(va_idx):,} | Test {len(te_idx):,}')
    print(f'  Features: {X_past.shape[2]} | X_future: {"ON" if has_future else "OFF"}')
    print(f'  Cap: {cap:.2f} (sqrt space)')
    print(f'  Val : 2024-09 → 2025-03 (flood 2024)')
    print(f'  Test: 2025-10 → 2025-12 (flood 2025) ← target evaluation')

    if len(te_idx) == 0:
        print(f'  [WARN] Test set RỖNG — data không có sau 2025-10-01!')
    else:
        # Kiểm tra timestamp range thực tế của test set
        te_ts = ts[te_idx]
        print(f'  Test range: {te_ts.min()} → {te_ts.max()}')

    return train_os, val_ds, test_ds, X_past.shape[2], has_future

print('Dataset classes defined  (test = Oct–Dec 2025)')

## 5. Training Function

In [ ]:
import copy

def train_one(name, cfg, data_dir=DATA_DIR, out_dir=OUT_DIR):
    print(f"\n{'='*60}")
    print(f"  {cfg['display_name']}  |  hidden={cfg['hidden_size']}  |  epochs={cfg['epochs']}")
    print(f"{'='*60}")

    train_os, val_ds, test_ds, input_size, has_future = load_reservoir(name, data_dir, cfg)

    QUANTILES = cfg['quantiles']
    BS        = cfg['batch_size']
    N_WORKERS = 2 if USE_AMP else 0

    train_loader = DataLoader(train_os, batch_size=BS, shuffle=True,
                              num_workers=N_WORKERS, pin_memory=USE_AMP,
                              persistent_workers=(N_WORKERS > 0))
    val_loader   = DataLoader(val_ds,   batch_size=BS, shuffle=False,
                              num_workers=N_WORKERS, pin_memory=USE_AMP,
                              persistent_workers=(N_WORKERS > 0))
    test_loader  = DataLoader(test_ds,  batch_size=BS, shuffle=False)

    n_future = cfg['n_future'] if has_future else 0
    model = PerReservoirModel(
        input_size=input_size,
        hidden_size=cfg['hidden_size'],
        horizon=cfg['horizon'],
        quantiles=QUANTILES,
        n_future=n_future,
    ).to(DEVICE)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Model params: {n_params:,}')

    optimizer  = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    amp_scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

    EPOCHS        = cfg['epochs']
    WARMUP_EPOCHS = cfg['warmup_epochs']
    PATIENCE      = cfg['patience']

    def lr_lambda(epoch):
        if epoch < WARMUP_EPOCHS:
            return float(epoch + 1) / float(WARMUP_EPOCHS)
        progress = (epoch - WARMUP_EPOCHS) / max(EPOCHS - WARMUP_EPOCHS, 1)
        return max(0.05, 0.5 * (1.0 + math.cos(math.pi * progress)))

    scheduler  = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    model_path = os.path.join(out_dir, f'{name}_best.pt')

    EMA_ALPHA   = 0.4
    ema_nse     = None
    best_nse    = float('-inf')
    patience_c  = 0
    history     = []
    t0          = time.time()

    # SWA: thu thap checkpoint tu 60% training tro di
    SWA_START  = int(EPOCHS * 0.6)
    swa_states = []

    for epoch in range(EPOCHS):
        # ── Train (voi noise augmentation) ───────────────────────────────────
        model.train()
        train_loss = 0.0
        for xb_past, xb_fut, yb in train_loader:
            xb_past = xb_past.to(DEVICE)
            yb      = yb.to(DEVICE)

            # Input noise: simulate cam bien khong chinh xac
            xb_past_aug = xb_past + torch.randn_like(xb_past) * 0.02

            if has_future:
                xb_fut = xb_fut.to(DEVICE)
                # Rain forecast noise: simulate sai so NWP (10-20% multiplicative)
                rain_scale  = 1.0 + torch.randn_like(xb_fut) * 0.15
                xb_fut_aug  = (xb_fut * rain_scale.clamp(min=0.1)).clamp(min=0.0)
            else:
                xb_fut_aug = None

            optimizer.zero_grad()
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                preds = model(xb_past_aug, xb_fut_aug)
                loss  = quantile_loss(preds, yb + torch.randn_like(yb) * 0.005, QUANTILES)
            amp_scaler.scale(loss).backward()
            amp_scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            amp_scaler.step(optimizer)
            amp_scaler.update()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        # ── Validate (flood season 2024, khong augment) ───────────────────────
        model.eval()
        val_loss, all_p, all_t = 0.0, [], []
        with torch.no_grad():
            for xb_past, xb_fut, yb in val_loader:
                xb_past = xb_past.to(DEVICE)
                yb      = yb.to(DEVICE)
                xb_fut  = xb_fut.to(DEVICE) if has_future else None
                with torch.amp.autocast('cuda', enabled=USE_AMP):
                    preds = model(xb_past, xb_fut)
                val_loss += quantile_loss(preds, yb, QUANTILES).item()
                all_p.append(preds.cpu()); all_t.append(yb.cpu())
        val_loss /= len(val_loader)
        mae, rmse, nse = compute_metrics(torch.cat(all_p), torch.cat(all_t))

        ema_nse = nse if ema_nse is None else EMA_ALPHA * nse + (1 - EMA_ALPHA) * ema_nse

        scheduler.step()
        lr = optimizer.param_groups[0]['lr']
        elapsed = (time.time() - t0) / 60
        history.append(dict(epoch=epoch+1, train=train_loss, val=val_loss,
                            mae=mae, rmse=rmse, nse=nse, ema_nse=ema_nse))

        # Luu best checkpoint
        marker = ''
        if ema_nse > best_nse:
            best_nse = ema_nse
            torch.save(model.state_dict(), model_path)
            patience_c = 0
            marker = f'  <- best (ema={ema_nse:.4f})'
        else:
            patience_c += 1

        # Thu thap checkpoint cho SWA (chi khi ema_nse > 0)
        if epoch >= SWA_START and ema_nse > 0.0:
            swa_states.append(copy.deepcopy(model.state_dict()))
            if len(swa_states) > 20:
                swa_states.pop(0)

        if (epoch + 1) % 5 == 0 or patience_c == 0:
            swa_mark = f' [swa:{len(swa_states)}]' if swa_states else ''
            print(f'  [{epoch+1:3d}/{EPOCHS}] {elapsed:5.1f}m | '
                  f'LR {lr:.5f} | Train {train_loss:.4f} | Val {val_loss:.4f} | '
                  f'NSE {nse:.4f}{marker}{swa_mark}')

        if patience_c >= PATIENCE:
            print(f'  Early stopping tai epoch {epoch+1}  (best ema_NSE={best_nse:.4f})')
            break

    # ── SWA: average checkpoint 60% cuoi training ────────────────────────────
    if len(swa_states) >= 5:
        print(f'\n  Applying SWA: averaging {len(swa_states)} checkpoints...')
        avg_state = {}
        for key in swa_states[0].keys():
            avg_state[key] = torch.stack([s[key].float() for s in swa_states]).mean(0)
        model.load_state_dict(avg_state)

        # Danh gia nhanh SWA tren val de so sanh
        model.eval()
        swa_p, swa_t = [], []
        with torch.no_grad():
            for xb_past, xb_fut, yb in val_loader:
                xb_past = xb_past.to(DEVICE)
                xb_fut  = xb_fut.to(DEVICE) if has_future else None
                swa_p.append(model(xb_past, xb_fut).cpu())
                swa_t.append(yb)
        _, _, swa_val_nse = compute_metrics(torch.cat(swa_p), torch.cat(swa_t))
        print(f'  SWA val NSE={swa_val_nse:.4f}  vs  Best single={best_nse:.4f}')

        if swa_val_nse >= best_nse * 0.97:  # dung SWA neu khong kem hon 3%
            torch.save(avg_state, model_path)
            print(f'  -> Su dung SWA model')
        else:
            model.load_state_dict(torch.load(model_path, map_location=DEVICE))
            print(f'  -> Giu lai best single checkpoint')
    else:
        model.load_state_dict(torch.load(model_path, map_location=DEVICE))
        print(f'\n  Loading best checkpoint (ema_NSE={best_nse:.4f})')

    # ── Test: flood season Oct-Dec 2025 ───────────────────────────────────────
    model.eval()
    all_p, all_t = [], []
    with torch.no_grad():
        for xb_past, xb_fut, yb in test_loader:
            xb_past = xb_past.to(DEVICE)
            xb_fut  = xb_fut.to(DEVICE) if has_future else None
            all_p.append(model(xb_past, xb_fut).cpu())
            all_t.append(yb)

    if len(all_p) == 0:
        print('  [WARN] Test set rong - khong co data Oct-Dec 2025!')
        return dict(name=name, best_nse=best_nse, history=history,
                    test_nse=float('nan'), test_mae=float('nan'), test_rmse=float('nan'))

    test_mae, test_rmse, test_nse = compute_metrics(torch.cat(all_p), torch.cat(all_t))
    print(f'\n  TEST Oct-Dec 2025:  NSE={test_nse:.4f}  MAE={test_mae:.2f}  RMSE={test_rmse:.2f}')
    print(f'  Saved: {model_path}')

    return dict(name=name, best_nse=best_nse, history=history,
                test_nse=test_nse, test_mae=test_mae, test_rmse=test_rmse)

print('train_one() defined  (+ input noise aug + rain forecast noise + SWA)')

## 6. Train All 4 Reservoirs

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
all_results = {}

for res_name, cfg in RESERVOIR_CONFIGS.items():
    result = train_one(res_name, cfg)
    all_results[res_name] = result

print('\nAll reservoirs trained!')

## 7. Results Summary

In [ ]:
rows = []
for name, r in all_results.items():
    rows.append({
        'Reservoir'    : RESERVOIR_CONFIGS[name]['display_name'],
        'NSE'          : round(r['test_nse'], 4),
        'MAE (m³/s)'   : round(r['test_mae'], 2),
        'RMSE (m³/s)'  : round(r['test_rmse'], 2),
        'Best Val NSE' : round(r['best_nse'], 5),
    })

df = pd.DataFrame(rows)
print('\n=== TEST 2025 RESULTS ===')
print(df.to_string(index=False))

df.to_csv(os.path.join(OUT_DIR, 'test_results.csv'), index=False)
print(f'\nSaved: {OUT_DIR}/test_results.csv')

## 8. Training History Plots

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, (name, r) in enumerate(all_results.items()):
    hist = r['history']
    epochs = [h['epoch'] for h in hist]
    ax = axes[i]
    ax.plot(epochs, [h['train'] for h in hist], label='Train Loss')
    ax.plot(epochs, [h['val']   for h in hist], label='Val Loss')
    ax2 = ax.twinx()
    ax2.plot(epochs, [h['nse']  for h in hist], color='green', linestyle='--', label='NSE (right)')
    ax2.set_ylabel('NSE', color='green')
    ax2.tick_params(axis='y', labelcolor='green')
    ax.set_title(RESERVOIR_CONFIGS[name]['display_name'])
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend(loc='upper left')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'training_history.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Saved: training_history.png')

## 9. Download Models

Sau khi notebook chạy xong, download các file sau từ `/kaggle/working/`:
```
avuong_best.pt
dakmi4_best.pt
songbung4_best.pt
songtranh2_best.pt
test_results.csv
training_history.png
```
Copy các `.pt` file vào `lstm_service/artifacts/` trên VPS.

In [ ]:
print('=== OUTPUT FILES ===')
for f in sorted(os.listdir(OUT_DIR)):
    size = os.path.getsize(os.path.join(OUT_DIR, f)) / 1e6
    print(f'  {f:<35} {size:7.2f} MB')